In [1]:
import json
import re
from pathlib import Path

import pandas as pd

In [3]:
RAW_DATA_PATH = Path("../data/raw")

geojson_files = sorted(RAW_DATA_PATH.glob("*.geojson"))

print("Number of GeoJSON files:", len(geojson_files))

for file in geojson_files:
    print(file.name)

Number of GeoJSON files: 20
new_delhi__2024-08-11_to_2024-08-11_.geojson
new_delhi__2024-08-12_to_2024-08-12_.geojson
new_delhi__2024-08-13_to_2024-08-13_.geojson
new_delhi__2024-08-14_to_2024-08-14_.geojson
new_delhi__2024-08-15_to_2024-08-15_.geojson
new_delhi__2024-08-16_to_2024-08-16_.geojson
new_delhi__2024-08-17_to_2024-08-17_.geojson
new_delhi__2024-08-18_to_2024-08-18_.geojson
new_delhi__2024-08-19_to_2024-08-19_.geojson
new_delhi__2024-08-20_to_2024-08-20_.geojson
new_delhi__2024-08-21_to_2024-08-21_.geojson
new_delhi__2024-08-22_to_2024-08-22_.geojson
new_delhi__2024-08-23_to_2024-08-23_.geojson
new_delhi__2024-08-24_to_2024-08-24_.geojson
new_delhi__2024-08-25_to_2024-08-25_.geojson
new_delhi__2024-08-26_to_2024-08-26_.geojson
new_delhi__2024-08-27_to_2024-08-27_.geojson
new_delhi__2024-08-28_to_2024-08-28_.geojson
new_delhi__2024-08-29_to_2024-08-29_.geojson
new_delhi__2024-08-30_to_2024-08-30_.geojson


## 2. Build a Reusable GeoJSON Processing Function

We create a function that converts one day's nested GeoJSON traffic data
into a flat tabular DataFrame.

In [6]:
def process_geojson(file_path):
    """
    Read one Delhi traffic GeoJSON file and convert it
    into a flat DataFrame with one row per road-hour observation.
    """

    # Extract date from filename
    date_match = re.search(r"2024-\d{2}-\d{2}", file_path.name)

    if date_match is None:
        raise ValueError(f"Could not extract date from {file_path.name}")

    date = date_match.group()

    # Load GeoJSON
    with open(file_path, "r") as f:
        data = json.load(f)

    rows = []

    # Process every feature
    for feature in data["features"]:
        properties = feature.get("properties", {})
        geometry = feature.get("geometry")

        # Skip metadata feature
        if geometry is None:
            continue

        # Road-level information
        segment_id = properties.get("segmentId")
        new_segment_id = properties.get("newSegmentId")
        speed_limit = properties.get("speedLimit")
        frc = properties.get("frc")
        street_name = properties.get("streetName")
        distance = properties.get("distance")

        # Hourly traffic observations
        for observation in properties.get("segmentProbeCounts", []):
            rows.append({
                "date": date,
                "segment_id": segment_id,
                "new_segment_id": new_segment_id,
                "speed_limit": speed_limit,
                "frc": frc,
                "street_name": street_name,
                "distance": distance,
                "time_set": observation.get("timeSet"),
                "probe_count": observation.get("probeCount")
            })

    return pd.DataFrame(rows)

In [8]:
test_df = process_geojson(geojson_files[0])

print(test_df.shape)
print(test_df["date"].unique())
print(test_df.head())

(598512, 9)
['2024-08-11']
         date      segment_id                         new_segment_id  \
0  2024-08-11 -13560111507837  -00004e30-3400-0400-0000-0000008bad86   
1  2024-08-11 -13560111507837  -00004e30-3400-0400-0000-0000008bad86   
2  2024-08-11 -13560111507837  -00004e30-3400-0400-0000-0000008bad86   
3  2024-08-11 -13560111507837  -00004e30-3400-0400-0000-0000008bad86   
4  2024-08-11 -13560111507837  -00004e30-3400-0400-0000-0000008bad86   

   speed_limit  frc        street_name  distance  time_set  probe_count  
0           45    6  Ghata Masjid Road     20.87         2            4  
1           45    6  Ghata Masjid Road     20.87         3            2  
2           45    6  Ghata Masjid Road     20.87         4            1  
3           45    6  Ghata Masjid Road     20.87         5            0  
4           45    6  Ghata Masjid Road     20.87         6            0  


In [9]:
for file in geojson_files:
    date_match = re.search(r"2024-\d{2}-\d{2}", file.name)
    print(file.name, "→", date_match.group())

new_delhi__2024-08-11_to_2024-08-11_.geojson → 2024-08-11
new_delhi__2024-08-12_to_2024-08-12_.geojson → 2024-08-12
new_delhi__2024-08-13_to_2024-08-13_.geojson → 2024-08-13
new_delhi__2024-08-14_to_2024-08-14_.geojson → 2024-08-14
new_delhi__2024-08-15_to_2024-08-15_.geojson → 2024-08-15
new_delhi__2024-08-16_to_2024-08-16_.geojson → 2024-08-16
new_delhi__2024-08-17_to_2024-08-17_.geojson → 2024-08-17
new_delhi__2024-08-18_to_2024-08-18_.geojson → 2024-08-18
new_delhi__2024-08-19_to_2024-08-19_.geojson → 2024-08-19
new_delhi__2024-08-20_to_2024-08-20_.geojson → 2024-08-20
new_delhi__2024-08-21_to_2024-08-21_.geojson → 2024-08-21
new_delhi__2024-08-22_to_2024-08-22_.geojson → 2024-08-22
new_delhi__2024-08-23_to_2024-08-23_.geojson → 2024-08-23
new_delhi__2024-08-24_to_2024-08-24_.geojson → 2024-08-24
new_delhi__2024-08-25_to_2024-08-25_.geojson → 2024-08-25
new_delhi__2024-08-26_to_2024-08-26_.geojson → 2024-08-26
new_delhi__2024-08-27_to_2024-08-27_.geojson → 2024-08-27
new_delhi__202

In [ ]:
## 3. Process All 20 Days

We now apply the validated processing function to every daily GeoJSON file
and combine the results into a single traffic dataset.

In [ ]:
## 3. Process All 20 Days

We now apply the validated processing function to every daily GeoJSON file
and combine the results into a single traffic dataset.

In [10]:
all_data = []

for i, file in enumerate(geojson_files, start=1):
    print(f"Processing {i}/{len(geojson_files)}: {file.name}")

    day_df = process_geojson(file)

    print(f"  Rows: {len(day_df):,}")

    all_data.append(day_df)

Processing 1/20: new_delhi__2024-08-11_to_2024-08-11_.geojson
  Rows: 598,512
Processing 2/20: new_delhi__2024-08-12_to_2024-08-12_.geojson
  Rows: 598,512
Processing 3/20: new_delhi__2024-08-13_to_2024-08-13_.geojson
  Rows: 598,512
Processing 4/20: new_delhi__2024-08-14_to_2024-08-14_.geojson
  Rows: 598,512
Processing 5/20: new_delhi__2024-08-15_to_2024-08-15_.geojson
  Rows: 598,512
Processing 6/20: new_delhi__2024-08-16_to_2024-08-16_.geojson
  Rows: 598,512
Processing 7/20: new_delhi__2024-08-17_to_2024-08-17_.geojson
  Rows: 598,512
Processing 8/20: new_delhi__2024-08-18_to_2024-08-18_.geojson
  Rows: 598,512
Processing 9/20: new_delhi__2024-08-19_to_2024-08-19_.geojson
  Rows: 598,512
Processing 10/20: new_delhi__2024-08-20_to_2024-08-20_.geojson
  Rows: 598,512
Processing 11/20: new_delhi__2024-08-21_to_2024-08-21_.geojson
  Rows: 598,512
Processing 12/20: new_delhi__2024-08-22_to_2024-08-22_.geojson
  Rows: 598,512
Processing 13/20: new_delhi__2024-08-23_to_2024-08-23_.geojso

In [11]:
df_all = pd.concat(all_data, ignore_index=True)

print("Final shape:", df_all.shape)

Final shape: (11970240, 9)


In [12]:
print("Date range:")
print(df_all["date"].min(), "→", df_all["date"].max())

Date range:
2024-08-11 → 2024-08-30


In [14]:
print("Unique days:", df_all["date"].nunique())

Unique days: 20


In [16]:
print("Unique roads:", df_all["segment_id"].nunique())

Unique roads: 24938


In [17]:
observations_per_road = (
    df_all.groupby("segment_id").size()
)

print(observations_per_road.describe())

count    24938.0
mean       480.0
std          0.0
min        480.0
25%        480.0
50%        480.0
75%        480.0
max        480.0
dtype: float64


In [18]:
# Convert date to datetime
df_all["date"] = pd.to_datetime(df_all["date"])

# Convert time_set (2-25) to hour (0-23)
df_all["hour"] = df_all["time_set"] - 2

# Create a proper timestamp
df_all["timestamp"] = (
    df_all["date"] +
    pd.to_timedelta(df_all["hour"], unit="h")
)

df_all[["date", "time_set", "hour", "timestamp", "probe_count"]].head(10)

,date,time_set,hour,timestamp,probe_count
0,2024-08-11,2,0,2024-08-11 00:00:00,4
1,2024-08-11,3,1,2024-08-11 01:00:00,2
2,2024-08-11,4,2,2024-08-11 02:00:00,1
3,2024-08-11,5,3,2024-08-11 03:00:00,0
4,2024-08-11,6,4,2024-08-11 04:00:00,0
5,2024-08-11,7,5,2024-08-11 05:00:00,0
6,2024-08-11,8,6,2024-08-11 06:00:00,0
7,2024-08-11,9,7,2024-08-11 07:00:00,2
8,2024-08-11,10,8,2024-08-11 08:00:00,2
9,2024-08-11,11,9,2024-08-11 09:00:00,4


In [19]:
print("Minimum timestamp:", df_all["timestamp"].min())
print("Maximum timestamp:", df_all["timestamp"].max())
print("Unique hours:", df_all["hour"].nunique())
print("Unique timestamps:", df_all["timestamp"].nunique())

Minimum timestamp: 2024-08-11 00:00:00
Maximum timestamp: 2024-08-30 23:00:00
Unique hours: 24
Unique timestamps: 480


In [20]:
# Create processed data directory
PROCESSED_DATA_PATH = Path("../data/processed")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Save the processed dataset in Parquet format
output_file = PROCESSED_DATA_PATH / "delhi_traffic_hourly.parquet"

df_all.to_parquet(
    output_file,
    index=False
)

print("Saved processed dataset to:")
print(output_file)
print(f"File size: {output_file.stat().st_size / (1024**2):.2f} MB")

Saved processed dataset to:
../data/processed/delhi_traffic_hourly.parquet
File size: 28.47 MB


In [21]:
# Load the saved Parquet file
df_check = pd.read_parquet(output_file)

print("Shape:", df_check.shape)
print("\nColumns:")
print(df_check.columns.tolist())

print("\nTimestamp range:")
print(df_check["timestamp"].min(), "→", df_check["timestamp"].max())

print("\nMemory usage:")
print(f"{df_check.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Shape: (11970240, 11)

Columns:
['date', 'segment_id', 'new_segment_id', 'speed_limit', 'frc', 'street_name', 'distance', 'time_set', 'probe_count', 'hour', 'timestamp']

Timestamp range:
2024-08-11 00:00:00 → 2024-08-30 23:00:00

Memory usage:
2684.75 MB
